# Pandas Advance - Comprehensive Guide\n## Performance Optimization, Large Datasets & Production-Ready Code\n\nIs notebook in we Pandas of advanced concepts cover perform  that real-world production systems in use are. this topics performance optimization, memory management, advanced I/O, and large-scale data processing par focus do are.\n

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom datetime import datetime\nimport time\nimport warnings\nwarnings.filterwarnings('ignore')\n\nprint('Pandas:', pd.__version__)\nprint('NumPy:', np.__version__)

---\n## 1. Performance Optimization\n\nPandas performance optimize karne of techniques taake aapka code fast and memory-efficient be.\n

### 1.1 Vectorization vs Loops\n\n**Golden Rule:** Kabhi also Python loops not use perform Pandas operations in. Vectorized operations 10x-1000x faster are.\n\nVectorization = NumPy of C-level optimizations benefit of uthana.\n

In [ ]:
# Performance comparison: loop vs Vectorized\nn = 100000\ndf = pd.DataFrame({'A': np.random.randn(n), 'B': np.random.randn(n)})\n\n# method 1: Python loop (slowest)\nstart = time.time()\nresult_loop = []\nfor i in range(len(df)):\n    result_loop.append(df['A'].iloc[i] + df['B'].iloc[i])\nloop_time = time.time() - start\nprint(f'Python loop: {loop_time:.4f} sec')\n\n# method 2: .apply() (slow)\nstart = time.time()\nresult_apply = df.apply(lambda row: row['A'] + row['B'], axis=1)\napply_time = time.time() - start\nprint(f'apply() with axis=1: {apply_time:.4f} sec')\n\n# method 3: Vectorized (fastest)\nstart = time.time()\nresult_vec = df['A'] + df['B']\nvec_time = time.time() - start\nprint(f'Vectorized (+ operator): {vec_time:.4f} sec')\n\nprint()\nprint(f'Vectorized is {loop_time/vec_time:.0f}x faster than loop')\nprint(f'Vectorized is {apply_time/vec_time:.0f}x faster than apply')

In [ ]:
# More vectorization examples\n# Complex condition (vectorized)\ndf['Category'] = np.where(df['A'] > 0, 'Positive', 'Negative')\ndf['Score'] = np.select(\n    [df['A'] > 1, df['A'] > 0, df['A'] > -1],\n    ['High', 'Medium', 'Low'],\n    default='Very Low'\n)\n\n# Vectorized string operations\ndf_str = pd.DataFrame({'Text': ['hello world', 'data science', 'python pandas'] * 30000})\n\nstart = time.time()\ndf_str['Upper'] = df_str['Text'].str.upper()\nprint(f'Vectorized str.upper(): {time.time() - start:.4f} sec')\n\nstart = time.time()\ndf_str['Upper_loop'] = [t.upper() for t in df_str['Text']]\nprint(f'loop str.upper(): {time.time() - start:.4f} sec')

### 1.2 Using .values / .to_numpy() for NumPy Operations\n\nKabhi kabhi Pandas DataFrame to NumPy array in convert karke operations to do faster is, especially when indexing overhead avoid to do be.\n

In [ ]:
# .values vs .to_numpy()\ndf = pd.DataFrame(np.random.randn(100000, 10), columns=list('ABCDEFGHIJ'))\n\n# Row-wise operation on DataFrame\nstart = time.time()\nresult = df.mean(axis=1)\nprint(f'DataFrame mean(axis=1): {time.time() - start:.4f} sec')\n\n# Convert to numpy and then compute\nstart = time.time()\nresult_np = df.to_numpy().mean(axis=1)\nprint(f'NumPy array mean(axis=1): {time.time() - start:.4f} sec')\nprint()\n\n# Complex row-wise operation on numpy\nstart = time.time()\n# DataFrame version\ndf_result = df.apply(lambda row: np.sqrt(row['A']**2 + row['B']**2), axis=1)\ndf_time = time.time() - start\nprint(f'apply() sqrt(A^2+B^2): {df_time:.4f} sec')\n\nstart = time.time()\n# NumPy version\narr = df[['A', 'B']].to_numpy()\nnp_result = np.sqrt(arr[:, 0]**2 + arr[:, 1]**2)\nnp_time = time.time() - start\nprint(f'NumPy sqrt: {np_time:.4f} sec')\nprint(f'NumPy is {df_time/np_time:.0f}x faster')\nprint()\n\n# Note: .values returns a view (no copy) for homogeneous dtypes\n# .to_numpy() is the recommended method (more explicit)

### 1.3 .itertuples() vs .iterrows()\n\nKabhi loop chalana hi pare to `itertuples()` use perform not of `iterrows()`. `itertuples()` named tuples return does is and ~10x faster is.\n

In [ ]:
# itertuples vs iterrows comparison\ndf = pd.DataFrame({\n    'A': np.random.randn(100000),\n    'B': np.random.randn(100000),\n    'C': np.random.randn(100000)\n})\n\n# iterrows (slow)\nstart = time.time()\nsums = []\nfor idx, row in df.iterrows():\n    sums.append(row['A'] + row['B'] + row['C'])\niterrows_time = time.time() - start\nprint(f'iterrows(): {iterrows_time:.4f} sec')\n\n# itertuples (fast)\nstart = time.time()\nsums = []\nfor row in df.itertuples():\n    sums.append(row.A + row.B + row.C)\nitertuples_time = time.time() - start\nprint(f'itertuples(): {itertuples_time:.4f} sec')\nprint(f'itertuples is {iterrows_time/itertuples_time:.0f}x faster')

In [ ]:
# itertuples with name=None (even faster - plain tuples)\nstart = time.time()\nsums = []\nfor row in df.itertuples(index=False, name=None):\n    sums.append(row[0] + row[1] + row[2])\ntuple_time = time.time() - start\nprint(f'itertuples(name=None): {tuple_time:.4f} sec')\nprint(f'Plain tuples are {itertuples_time/tuple_time:.0f}x faster than named')

### 1.4 df.apply() Optimization with raw=True\n\n`raw=True` parameter apply to NumPy arrays pass does is (Series of bajaye), that faster is.\n

In [ ]:
# apply() raw=True optimization\ndf = pd.DataFrame(np.random.randn(10000, 4), columns=list('ABCD'))\n\ndef my_func(row):\n    return row['A'] * row['B'] + np.sqrt(abs(row['C'])) * row['D']\n\ndef my_func_raw(arr):\n    return arr[0] * arr[1] + np.sqrt(abs(arr[2])) * arr[3]\n\n# Without raw (Series passed)\nstart = time.time()\nresult1 = df.apply(my_func, axis=1)\nt1 = time.time() - start\nprint(f'apply(axis=1) default: {t1:.4f} sec')\n\n# With raw=True (NumPy array passed)\nstart = time.time()\nresult2 = df.apply(my_func_raw, axis=1, raw=True)\nt2 = time.time() - start\nprint(f'apply(axis=1, raw=True): {t2:.4f} sec')\nprint(f'raw=True is {t1/t2:.0f}x faster')

### 1.5 Using Numba with Pandas\n\nNumba JIT compilation of uses is Python code to machine code in convert karne for. loop-heavy operations for bohot effective.\n

In [ ]:
# Numba optimization\ntry:\n    from numba import jit, vectorize\n    import numba\n    print('Numba version:', numba.__version__)\n    \n    df = pd.DataFrame(np.random.randn(100000, 3), columns=list('ABC'))\n    \n    # Pure Python function\n    def compute_python(a, b, c):\n        result = np.zeros_like(a)\n        for i in range(len(a)):\n            result[i] = a[i] * b[i] + np.sqrt(abs(c[i]))\n        return result\n    \n    # Numba JIT compiled\n    @jit(nopython=True)\n    def compute_numba(a, b, c):\n        result = np.zeros_like(a)\n        for i in range(len(a)):\n            result[i] = a[i] * b[i] + np.sqrt(abs(c[i]))\n        return result\n    \n    # Warm up JIT\n    a = df['A'].to_numpy()\n    b = df['B'].to_numpy()\n    c = df['C'].to_numpy()\n    _ = compute_numba(a, b, c)\n    \n    # Python version\n    start = time.time()\n    result1 = compute_python(a, b, c)\n    t1 = time.time() - start\n    print(f'Pure Python loop: {t1:.4f} sec')\n    \n    # Numba version\n    start = time.time()\n    result2 = compute_numba(a, b, c)\n    t2 = time.time() - start\n    print(f'Numba JIT (nopython): {t2:.4f} sec')\n    print(f'Numba is {t1/t2:.0f}x faster')\n    \n    # Vectorized (baseline)\n    start = time.time()\n    result3 = a * b + np.sqrt(np.abs(c))\n    t3 = time.time() - start\n    print(f'Vectorized NumPy: {t3:.4f} sec')\n    \nexcept ImportError:\n    print('Numba not installed. install with: pip install numba')

### 1.6 Chaining vs Intermediate Variables\n\nmethod chaining readable is but intermediate variables debugging in help do are. Performance in koi significant farq no.\n

In [ ]:
# Chaining vs intermediate variables\ndf = pd.DataFrame({\n    'Category': np.random.choice(['A', 'B', 'C'], 10000),\n    'value': np.random.randn(10000),\n    'Weight': np.random.randn(10000)\n})\n\n# method chaining (Pandas idiomatic)\nstart = time.time()\nresult_chain = (\n    df.assign(\n        Score=lambda x: x['value'] * x['Weight'],\n        Quartile=lambda x: pd.qcut(x['value'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])\n    )\n    .query('Score > 0')\n    .groupby(['Category', 'Quartile'])['Score']\n    .agg(['mean', 'count', 'sum'])\n    .sort_values('mean', ascending=False)\n)\nchain_time = time.time() - start\nprint(f'method chaining: {chain_time:.4f} sec')\nprint(result_chain)\nprint()\n\n# Intermediate variables\nstart = time.time()\ndf_temp = df.copy()\ndf_temp['Score'] = df_temp['value'] * df_temp['Weight']\ndf_temp['Quartile'] = pd.qcut(df_temp['value'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])\ndf_temp = df_temp[df_temp['Score'] > 0]\nresult_temp = df_temp.groupby(['Category', 'Quartile'])['Score'].agg(['mean', 'count', 'sum'])\nresult_temp = result_temp.sort_values('mean', ascending=False)\ninter_time = time.time() - start\nprint(f'Intermediate variables: {inter_time:.4f} sec')\nprint(result_temp)

---\n## 2. Memory Optimization\n\nLarge datasets with work do waqt memory optimization bohot important is. Pandas default data types memory-efficient no are.\n

### 2.1 Downcasting Numeric Types\n\nBy default Pandas int64 and float64 uses is. if aapki values int8/16/32 ya float32/16 in fit be can are to memory 50-75% bacha can are.\n

In [ ]:
# Memory optimization with downcasting\nn = 100000\ndf = pd.DataFrame({\n    'ID': range(n),\n    'Age': np.random.randint(18, 80, n),\n    'Salary': np.random.randint(30000, 100000, n).astype(float),\n    'Score': np.random.uniform(0, 100, n),\n    'Category': np.random.choice(['A', 'B', 'C', 'D', 'E'], n)\n})\n\nprint('Before optimization:')\nprint(df.dtypes)\nprint(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')\nprint()\n\n# Optimize numeric columns\ndf['ID'] = pd.to_numeric(df['ID'], downcast='integer')\ndf['Age'] = pd.to_numeric(df['Age'], downcast='integer')\ndf['Salary'] = pd.to_numeric(df['Salary'], downcast='float')\ndf['Score'] = pd.to_numeric(df['Score'], downcast='float')\ndf['Category'] = df['Category'].astype('category')\n\nprint('After optimization:')\nprint(df.dtypes)\nprint(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')\nprint(f'Saved: {100 * (1 - df.memory_usage(deep=True).sum() / (n * 8 * 5 + n * 8 * 4)):.0f}%')

In [ ]:
# Automatic memory optimization function\ndef optimize_memory(df):\n    start_mem = df.memory_usage(deep=True).sum()\n    \n    for col in df.columns:\n        col_type = df[col].dtype\n        \n        if col_type != 'object' and str(col_type)[:3] != 'cat':\n            c_min = df[col].min()\n            c_max = df[col].max()\n            \n            if str(col_type)[:3] == 'int' or str(col_type)[:4] == 'uint':\n                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:\n                    df[col] = df[col].astype(np.int8)\n                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:\n                    df[col] = df[col].astype(np.int16)\n                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:\n                    df[col] = df[col].astype(np.int32)\n            elif str(col_type)[:5] == 'float':\n                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:\n                    df[col] = df[col].astype(np.float32)\n        elif col_type == 'object':\n            if df[col].nunique() / len(df) < 0.5:\n                df[col] = df[col].astype('category')\n    \n    end_mem = df.memory_usage(deep=True).sum()\n    reduction = (1 - end_mem / start_mem) * 100\n    print(f'Memory: {start_mem/1024**2:.2f} MB -> {end_mem/1024**2:.2f} MB ({reduction:.0f}% reduction)')\n    return df\n\n# Test the function\ndf_large = pd.DataFrame({\n    'ID': range(500000),\n    'value': np.random.randn(500000),\n    'Category': np.random.choice(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'], 500000)\n})\ndf_opt = optimize_memory(df_large)

### 2.2 Using Categorical for String Columns\n\nString columns jin in limited unique values hon unhe category in convert to do memory drastically reduce does is.\n

In [ ]:
# Categorical memory comparison\nn = 200000\ndf_str = pd.DataFrame({\n    'City': np.random.choice(['Karachi', 'Lahore', 'Islamabad', 'Peshawar', 'Quetta'], n),\n    'Country': 'Pakistan',\n    'Product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Watch', 'Headphones', 'Charger'], n),\n    'Long_Text': ['This is a long text string for testing purposes'] * n\n})\n\nprint('Memory with object dtype:')\nprint(df_str.memory_usage(deep=True) / 1024**2)\nprint(f'Total: {df_str.memory_usage(deep=True).sum()/1024**2:.2f} MB')\nprint()\n\n# Convert to categorical\nfor col in ['City', 'Country', 'Product']:\n    df_str[col] = df_str[col].astype('category')\n\nprint('Memory with category dtype:')\nprint(df_str.memory_usage(deep=True) / 1024**2)\nprint(f'Total: {df_str.memory_usage(deep=True).sum()/1024**2:.2f} MB')

### 2.3 Sparse Data Structures\n\nSparse structures then use perform when aapke data in mostly zeros ya missing values hon. Pandas SparseArray and SparseDataFrame is for are.\n

In [ ]:
# Sparse data example\nfrom pandas.arrays import SparseArray\n\n# Create sparse array (95% zeros)\ndense_arr = np.random.choice([0, 1], size=100000, p=[0.95, 0.05])\nsparse_arr = SparseArray(dense_arr, fill_value=0)\n\nprint('Dense array memory:', dense_arr.nbytes, 'bytes')\nprint('Sparse array memory:', sparse_arr.memory_usage(), 'bytes')\nprint(f'Sparse is {dense_arr.nbytes / sparse_arr.memory_usage():.0f}x smaller')\nprint()\n\ndf_dense = pd.DataFrame({'A': dense_arr, 'B': dense_arr * 2})\ndf_sparse = pd.DataFrame({'A': SparseArray(dense_arr, fill_value=0),\n                          'B': SparseArray(dense_arr * 2, fill_value=0)})\n\nprint('Dense DF memory:', df_dense.memory_usage(deep=True).sum(), 'bytes')\nprint('Sparse DF memory:', df_sparse.memory_usage(deep=True).sum(), 'bytes')

### 2.4 Chunking Large Datasets (chunksize)\n\nwhen dataset memory in fit not be to chunks in read do, process do, and combine do.\n

In [ ]:
# Chunking example (simulated)\n# First, create a large CSV\nlarge_df = pd.DataFrame({'A': range(1000000), 'B': np.random.randn(1000000)})\nlarge_df.to_csv('large_file.csv', index=False)\nprint('Created large_file.csv')\nprint()\n\n# Read in chunks and process\nchunk_size = 50000\ntotal_sum = 0\ntotal_count = 0\n\nstart = time.time()\nfor chunk in pd.read_csv('large_file.csv', chunksize=chunk_size):\n    total_sum += chunk['B'].sum()\n    total_count += len(chunk)\n\nprint(f'Chunked processing complete')\nprint(f'Total sum: {total_sum:.2f}, Count: {total_count}')\nprint(f'Time: {time.time() - start:.2f} sec')\nprint()\n\n# Read entire file for comparison\nstart = time.time()\nfull_df = pd.read_csv('large_file.csv')\nprint(f'Full read time: {time.time() - start:.2f} sec')\nprint(f'Full memory: {full_df.memory_usage(deep=True).sum()/1024**2:.2f} MB')

In [ ]:
# Advanced chunking: process and save intermediate results\nchunk_results = []\n\nfor chunk in pd.read_csv('large_file.csv', chunksize=100000):\n    # Process each chunk\n    chunk['C'] = chunk['A'] * chunk['B']\n    chunk['Group'] = pd.cut(chunk['A'], bins=10, labels=range(10))\n    result = chunk.groupby('Group')['C'].sum()\n    chunk_results.append(result)\n\n# Combine results\nfinal_result = pd.concat(chunk_results).groupby(level=0).sum()\nprint('Chunked aggregation result:')\nprint(final_result)

### 2.5 Dtype Specification\n\nread_csv in dtype specify karke memory and parsing time both bacha can are.\n

In [ ]:
# Dtype specification in read_csv\n# Without dtype spec\nstart = time.time()\ndf_default = pd.read_csv('large_file.csv')\nt1 = time.time() - start\nprint(f'Default read: {t1:.2f} sec, types: {df_default.dtypes.tolist()}')\n\n# With dtype spec\nstart = time.time()\ndf_typed = pd.read_csv('large_file.csv', dtype={'A': np.int32, 'B': np.float32})\nt2 = time.time() - start\nprint(f'Typed read: {t2:.2f} sec, types: {df_typed.dtypes.tolist()}')\nprint(f'Memory: {df_default.memory_usage(deep=True).sum()/1024**2:.2f} MB vs {df_typed.memory_usage(deep=True).sum()/1024**2:.2f} MB')

---\n## 3. Advanced Reshaping\n

### 3.1 pd.wide_to_long()\n\nWide format with long format in convert does is when columns in stubnames hon.\n

In [ ]:
# wide_to_long example\ndf = pd.DataFrame({\n    'ID': [1, 2],\n    'Name': ['Ali', 'Sara'],\n    'Math_2022': [85, 90],\n    'Math_2023': [88, 92],\n    'Sci_2022': [79, 85],\n    'Sci_2023': [82, 88],\n    'Eng_2022': [90, 88],\n    'Eng_2023': [92, 91]\n})\nprint('Wide format:')\nprint(df)\nprint()\n\n# Convert to long format\nlong_df = pd.wide_to_long(df, stubnames=['Math', 'Sci', 'Eng'],\n                            i='ID', j='Year', sep='_', suffix='\\d+')\nprint('Long format:')\nprint(long_df)

### 3.2 Advanced Pivot and Melt\n

In [ ]:
# Advanced pivot/melt\ndf = pd.DataFrame({\n    'Product': ['A', 'A', 'B', 'B'],\n    'Metric': ['Sales', 'Profit', 'Sales', 'Profit'],\n    'Q1': [100, 20, 150, 30],\n    'Q2': [120, 25, 170, 35],\n    'Q3': [110, 22, 160, 32],\n    'Q4': [130, 28, 180, 38]\n})\nprint('Original:')\nprint(df)\nprint()\n\n# Melt: wide to long\nmelted = df.melt(id_vars=['Product', 'Metric'], var_name='Quarter', value_name='value')\nprint('Melted:')\nprint(melted)\nprint()\n\n# Pivot: long to wide (multiple value columns)\npivoted = melted.pivot_table(index=['Product', 'Quarter'],\n                              columns='Metric', values='value').reset_index()\nprint('Pivoted back:')\nprint(pivoted)

### 3.3 Explode (Lists into Rows)\n\nwhen ek cell in list be and it multiple rows in todna be.\n

In [ ]:
# Explode examples\ndf = pd.DataFrame({\n    'ID': [1, 2, 3],\n    'Name': ['Ali', 'Sara', 'Ahmed'],\n    'Subjects': [['Math', 'Science'], ['English'], ['Math', 'Science', 'English']],\n    'Scores': [[85, 79], [90], [78, 92, 88]]\n})\nprint('Data with lists:')\nprint(df)\nprint()\n\n# Explode a single column\nprint('Exploded Subjects:')\nprint(df.explode('Subjects'))\nprint()\n\n# Explode multiple columns (must have same length lists)\nprint('Exploded both Subjects and Scores:')\nprint(df.explode(['Subjects', 'Scores']))\nprint()\n\n# Explode with string containing lists\ndf_tags = pd.DataFrame({\n    'Post': [1, 2],\n    'Tags': ['python pandas data', 'machine learning ai']\n})\nprint('Split and explode:')\nprint(df_tags.assign(Tags=df_tags['Tags'].str.split()).explode('Tags'))

### 3.4 Cross-Join (Cartesian Product)\n\nevery row of left DataFrame to every row of right DataFrame with join does is.\n

In [ ]:
# Cross-join\nproducts = pd.DataFrame({'Product': ['Laptop', 'Phone'], 'Price': [50000, 15000]})\ncities = pd.DataFrame({'City': ['Karachi', 'Lahore', 'Islamabad'], 'Tax_Rate': [0.13, 0.14, 0.12]})\n\nprint('Products:')\nprint(products)\nprint()\nprint('Cities:')\nprint(cities)\nprint()\n\n# method 1: merge with how='cross' (Pandas 1.2+)\ncross1 = products.merge(cities, how='cross')\nprint('Cross join (merge how=cross):')\nprint(cross1)\nprint()\n\n# method 2: assign key\nproducts['key'] = 1\ncities['key'] = 1\ncross2 = products.merge(cities, on='key').drop('key', axis=1)\nprint('Cross join (key method):')\nprint(cross2)

---\n## 4. Advanced Apply and Vectorization\n

In [ ]:
# apply() with axis=1 (row-wise)\ndf = pd.DataFrame({\n    'A': [1, 2, 3],\n    'B': [4, 5, 6],\n    'C': [7, 8, 9]\n})\n\ndef row_operation(row):\n    if row['A'] > row['B']:\n        return row['A'] * row['C']\n    else:\n        return row['B'] * row['C']\n\nprint('Row-wise apply:')\ndf['Result'] = df.apply(row_operation, axis=1)\nprint(df)\nprint()\n\n# applymap() for element-wise operations\nprint('applymap() - element wise:')\nprint(df[['A', 'B', 'C']].applymap(lambda x: x ** 2))\nprint()\n\n# pipe() for method chaining\ndef add_features(df):\n    return df.assign(sum=df['A'] + df['B'] + df['C'], Product=df['A'] * df['B'] * df['C'])\n\ndef filter_results(df):\n    return df[df['sum'] > 10]\n\nresult = (df.pipe(add_features).pipe(filter_results))\nprint('pipe() chaining:')\nprint(result)

In [ ]:
# Custom functions with groupby().apply()\ndf = pd.DataFrame({\n    'Group': np.random.choice(['A', 'B', 'C'], 30),\n    'X': np.random.randn(30),\n    'Y': np.random.randn(30)\n})\n\ndef fit_line(group):\n    # Fit a simple linear regression per group\n    x = group['X'].values\n    y = group['Y'].values\n    slope = np.cov(x, y)[0, 1] / np.var(x) if np.var(x) > 0 else 0\n    intercept = y.mean() - slope * x.mean()\n    r2 = np.corrcoef(x, y)[0, 1]**2 if len(x) > 2 else 0\n    return pd.Series({'Slope': slope, 'Intercept': intercept, 'R2': r2})\n\nprint('Group-wise regression:')\nprint(df.groupby('Group').apply(fit_line))\nprint()\n\n# Swifter for parallel apply\ntry:\n    import swifter\n    print('Swifter available - parallel apply possible')\n    start = time.time()\n    result = df.groupby('Group').swifter.apply(fit_line)\n    print(f'Swifter apply: {time.time() - start:.4f} sec')\nexcept ImportError:\n    print('Swifter not installed. pip install swifter')

---\n## 5. Advanced I/O\n\nProduction in different file formats with work to do padta is. every format of apne pros and cons are.\n

In [ ]:
# Compare file formats\ndf = pd.DataFrame(np.random.randn(100000, 10), columns=list('ABCDEFGHIJ'))\ndf['Category'] = np.random.choice(['Cat1', 'Cat2', 'Cat3'], 100000)\n\nformats = []\n\n# CSV\nstart = time.time()\ndf.to_csv('test.csv', index=False)\ncsv_write = time.time() - start\nimport os\ncsv_size = os.path.getsize('test.csv') / 1024**2\nstart = time.time()\npd.read_csv('test.csv')\ncsv_read = time.time() - start\nformats.append(['CSV', csv_size, csv_write, csv_read])\nprint(f'CSV: {csv_size:.1f}MB, Write: {csv_write:.3f}s, Read: {csv_read:.3f}s')\n\n# Parquet (if installed)\ntry:\n    start = time.time()\n    df.to_parquet('test.parquet', index=False)\n    pq_write = time.time() - start\n    pq_size = os.path.getsize('test.parquet') / 1024**2\n    start = time.time()\n    pd.read_parquet('test.parquet')\n    pq_read = time.time() - start\n    formatters.append(['Parquet', pq_size, pq_write, pq_read])\n    print(f'Parquet: {pq_size:.1f}MB, Write: {pq_write:.3f}s, Read: {pq_read:.3f}s')\nexcept exception as e:\n    print(f'Parquet not available: {e}')\n\n# Feather (if installed)\ntry:\n    start = time.time()\n    df.to_feather('test.feather')\n    fw_write = time.time() - start\n    fw_size = os.path.getsize('test.feather') / 1024**2\n    start = time.time()\n    pd.read_feather('test.feather')\n    fw_read = time.time() - start\n    formatters.append(['Feather', fw_size, fw_write, fw_read])\n    print(f'Feather: {fw_size:.1f}MB, Write: {fw_write:.3f}s, Read: {fw_read:.3f}s')\nexcept exception as e:\n    print(f'Feather not available: {e}')

In [ ]:
# Parquet format benefits (columnar storage, compressed, fast I/O)\nprint('=== Parquet Features ===')\nprint('1. Columnar storage - sirf required columns read can')\nprint('2. Compression - smaller file size (snappy, gzip, etc.)')\nprint('3. Schema preservation - data types retain are')\nprint('4. Partitioning - large datasets to partition can')\nprint()\n\n# Read only specific columns from parquet\ntry:\n    df_pq = pd.read_parquet('test.parquet', columns=['A', 'B', 'Category'])\n    print('Read only A, B, Category from parquet:')\n    print(df_pq.head())\nexcept exception as e:\n    print('Parquet not available:', e)

In [ ]:
# HDF5 format\n# try:\n#     store = pd.HDFStore('test.h5')\n#     store.put('data', df, format='table', data_columns=True)\n#     store.close()\n#     print('HDF5 saved')\n#     \n#     # Query from HDF5\n#     df_hdf = pd.read_hdf('test.h5', 'data', where='index < 1000')\n#     print(f'HDF5 query result: {len(df_hdf)} rows')\n# except exception as e:\n#     print('HDF5:', e)\n\n# Reading HTML tables from web\n# try:\n#     tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_countries_by_population')\n#     print(f'Found {len(tables)} tables on the page')\n#     print(tables[0].head())\n# except exception as e:\n#     print('HTML tables:', e)\n\n# Reading from clipboard\n# df_clip = pd.read_clipboard()  # Copies from system clipboard\n\n# Reading from databases with SQLAlchemy\n# from sqlalchemy import create_engine\n# engine = create_engine('sqlite:///mydb.sqlite')\n# df_sql = pd.read_sql('SELECT * from table', engine)\n# df_sql = pd.read_sql_table('table', engine)\n# df_sql = pd.read_sql_query('SELECT * from table WHERE col > 10', engine)

---\n## 6. Advanced Time Series\n

In [ ]:
# Custom Business Days\nfrom pandas.tseries.offsets import CustomBusinessDay, CustomBusinessMonthEnd\nfrom pandas.tseries.holiday import AbstractHolidayCalendar, Holiday, nearest_workday\n\n# Define Pakistan holidays\nclass PakistanHolidays(AbstractHolidayCalendar):\n    rules = [\n        Holiday('Pakistan Day', month=3, day=23, observance=nearest_workday),\n        Holiday('Labour Day', month=5, day=1, observance=nearest_workday),\n        Holiday('Independence Day', month=8, day=14, observance=nearest_workday),\n        Holiday('Quaid-e-Azam Day', month=12, day=25, observance=nearest_workday)\n    ]\n\npk_bd = CustomBusinessDay(calendar=PakistanHolidays())\n\nprint('Pakistan business days (2023-01-01 to 2023-01-15):')\nprint(pd.date_range('2023-01-01', '2023-01-15', freq=pk_bd))\nprint()\n\n# Period and PeriodIndex\nprint('Period and PeriodIndex:')\np = pd.Period('2023-01', freq='M')\nprint('Period:', p)\nprint('Start time:', p.start_time)\nprint('End time:', p.end_time)\nprint()\n\nperiod_idx = pd.period_range('2023-01', periods=12, freq='M')\nprint('Period range:', period_idx)\nprint()\n\ndf_period = pd.DataFrame({\n    'Sales': np.random.randint(100, 500, 12)\n}, index=period_idx)\nprint('DataFrame with PeriodIndex:')\nprint(df_period)

In [ ]:
# Time series decomposition (trend, seasonal, residual)\nfrom statsmodels.tsa.seasonal import seasonal_decompose\n\nnp.random.seed(42)\ndates = pd.date_range('2022-01-01', '2023-12-31', freq='D')\n\n# Create synthetic time series with trend + seasonality + noise\ntrend = np.linspace(100, 200, len(dates))\nseasonal = 20 * np.sin(2 * np.pi * np.arange(len(dates)) / 365)  # yearly cycle\nweekly = 10 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)  # weekly cycle\nnoise = np.random.randn(len(dates)) * 5\nts = trend + seasonal + weekly + noise\n\ndf_ts = pd.DataFrame({'value': ts}, index=dates)\nprint('Time series data (first 5):')\nprint(df_ts.head())\nprint()\n\n# Decompose (weekly - 7 days)\ntry:\n    decomposition = seasonal_decompose(df_ts['value'], model='additive', period=7)\n    fig, axes = plt.subplots(4, 1, figsize=(14, 10))\n    df_ts['value'].plot(ax=axes[0], title='Original')\n    decomposition.trend.plot(ax=axes[1], title='Trend')\n    decomposition.seasonal.plot(ax=axes[2], title='Seasonal')\n    decomposition.resid.plot(ax=axes[3], title='Residual')\n    plt.tight_layout()\n    plt.show()\nexcept ImportError:\n    print('statsmodels not available. install: pip install statsmodels')

In [ ]:
# Rolling correlations and covariances\ndf_ts['Rolling_Corr'] = df_ts['value'].rolling(30).corr(\n    df_ts['value'].shift(1))  # correlation with previous day\n\ndf_ts['Rolling_Cov'] = df_ts['value'].rolling(30).cov(df_ts['value'].shift(1))\nprint('Rolling statistics (last 10):')\nprint(df_ts[['value', 'Rolling_Corr', 'Rolling_Cov']].tail(10))\nprint()\n\n# Advanced resampling\n# Custom resampling: group by day of week\nprint('Average by day of week:')\nprint(df_ts.groupby(df_ts.index.dayofweek)['value'].mean())\nprint()\n\n# Resample with custom label and closed end\nprint('Resample with custom label:')\nprint(df_ts.resample('W', label='right', closed='right').mean().head())

---\n## 7. Advanced GroupBy with Custom Functions\n

In [ ]:
# GroupBy with nlargest and nsmallest\ndf = pd.DataFrame({\n    'Category': np.random.choice(['A', 'B', 'C'], 50),\n    'value': np.random.randint(1, 100, 50),\n    'Weight': np.random.uniform(0.5, 2.0, 50)\n})\n\nprint('Top 3 values per category:')\nprint(df.groupby('Category')['value'].nlargest(3))\nprint()\n\n# Better approach: return full rows for largest\ndef top_n(group, n=2):\n    return group.nlargest(n, 'value')\n\nprint('Top 2 full rows per category:')\nprint(df.groupby('Category').apply(top_n, n=2))\nprint()\n\n# GroupBy with rank\ndf['Rank_Dense'] = df.groupby('Category')['value'].rank(method='dense', ascending=False)\ndf['Rank_Pct'] = df.groupby('Category')['value'].rank(pct=True)\nprint('Rank within groups:')\nprint(df.sort_values(['Category', 'Rank_Dense']).head(10))

In [ ]:
# GroupBy with custom window functions\n# Rolling rank within groups\ndf_grouped = df.groupby('Category')\n\ndef rolling_rank(group):\n    return group['value'].rolling(3, min_periods=1).apply(\n        lambda x: pd.Series(x).rank().iloc[-1]\n    )\n\ndf['Rolling_Rank'] = df_grouped.apply(rolling_rank).reset_index(level=0, drop=True)\nprint('Rolling rank within groups:')\nprint(df.sort_values(['Category']).head(15))\nprint()\n\n# GroupBy apply returning DataFrames\ndef add_group_stats(group):\n    group['Group_Mean'] = group['value'].mean()\n    group['Group_Std'] = group['value'].std()\n    group['Z_Score'] = (group['value'] - group['Group_Mean']) / group['Group_Std']\n    return group\n\nresult = df.groupby('Category').apply(add_group_stats)\nprint('GroupBy apply returning DataFrame:')\nprint(result.head(10))

---\n## 8. Styler and Visualization\n\nPandas Styler with DataFrame to visually appealing make can are. Conditional formatting, color gradients, bar charts.\n

In [ ]:
# Styler basics\ndf = pd.DataFrame({\n    'Student': ['Ali', 'Sara', 'Ahmed', 'Fatima', 'Usman'],\n    'Math': [85, 90, 78, 92, 88],\n    'Science': [79, 85, 92, 88, 76],\n    'English': [90, 88, 75, 95, 82],\n    'Total': [254, 263, 245, 275, 246]\n})\nprint('Original data:')\nprint(df)\nprint()\n\n# Style: highlight max\nstyled = df.style.highlight_max(color='lightgreen', axis=0)\nprint('Styled (highlight max) - displayed in notebook')

In [ ]:
# Various Styler options\n# Highlight min\nstyled_min = df.style.highlight_min(color='lightcoral', axis=0)\n\n# Color gradient (heatmap)\nstyled_grad = df[['Math', 'Science', 'English', 'Total']].style.background_gradient(cmap='Blues')\n\n# Bar chart in cells\nstyled_bar = df[['Math', 'Science', 'English']].style.bar(subset=['Math', 'Science', 'English'],\n                                                           color=['lightcoral', 'lightgreen'])\n\n# Combine multiple styles\ncombined_style = (df.style\n    .background_gradient(subset=['Math', 'Science', 'English'], cmap='viridis')\n    .bar(subset=['Total'], color='skyblue')\n    .format({'Total': '${:,.0f}'})\n)\nprint('Combined styling applied')

In [ ]:
# Export styled DataFrame to Excel\ntry:\n    with pd.ExcelWriter('styled_output.xlsx', engine='openpyxl') as writer:\n        (df.style\n         .background_gradient(subset=['Math', 'Science', 'English'], cmap='RdYlGn')\n         .bar(subset=['Total'], color='skyblue')\n         .to_excel(writer, sheet_name='Grades', index=False)\n        )\n    print('Styled Excel saved to styled_output.xlsx')\nexcept exception as e:\n    print('error saving styled Excel:', e)\n\n# Custom styling functions\ndef color_high_score(val):\n    color = 'green' if val >= 90 else 'orange' if val >= 80 else 'red'\n    return f'color: {color}; font-weight: bold'\n\nstyled_custom = df[['Math', 'Science', 'English']].style.applymap(color_high_score)\nprint('Custom styling function applied')

---\n## 9. MultiIndex Advanced\n

In [ ]:
# Advanced MultiIndex slicing with IndexSlice\nidx = pd.IndexSlice\n\n# Create complex MultiIndex\narrays = [\n    ['North', 'North', 'North', 'South', 'South', 'South', 'East', 'East', 'East'],\n    ['Laptop', 'Phone', 'Tablet', 'Laptop', 'Phone', 'Tablet', 'Laptop', 'Phone', 'Tablet'],\n    [2021, 2021, 2021, 2022, 2022, 2022, 2021, 2021, 2021]\n]\nmi = pd.MultiIndex.from_arrays(arrays, names=['Region', 'Product', 'Year'])\n\ndf_mi = pd.DataFrame({\n    'Sales': np.random.randint(100, 500, 9),\n    'Profit': np.random.randint(10, 100, 9)\n}, index=mi)\n\nprint('MultiIndex DataFrame:')\nprint(df_mi)\nprint()\n\n# Advanced slicing\nprint('Slice: North region:')\nprint(df_mi.loc['North'])\nprint()\n\nprint('Slice: North + Laptop:')\nprint(df_mi.loc[('North', 'Laptop')])\nprint()\n\nprint('Slice with IndexSlice:')\nprint(df_mi.sort_index().loc[idx['North':'South', :, 2021], :])\nprint()\n\n# Sorting MultiIndex\nprint('sorted MultiIndex:')\nprint(df_mi.sort_index())\nprint()\n\n# GroupBy on MultiIndex levels\nprint('sum by Region and Product:')\nprint(df_mi.groupby(level=['Region', 'Product']).sum())\nprint()\n\n# Reordering MultiIndex levels\nprint('Reordered levels:')\nprint(df_mi.reorder_levels(['Year', 'Region', 'Product']).sort_index())\nprint()\n\n# Swap levels\nprint('Swapped levels:')\nprint(df_mi.swaplevel('Region', 'Product'))

---\n## 10. Chaining Best Practices\n

In [ ]:
# method chaining best practices\ndf = pd.DataFrame({\n    'Name': ['Ali', 'Sara', 'Ahmed', 'Fatima', 'Usman'],\n    'Dept': ['IT', 'HR', 'IT', 'HR', 'IT'],\n    'Salary': [50000, 60000, 55000, 70000, 48000],\n    'Experience': [2, 3, 5, 1, 4]\n})\n\n# Chain with .assign(), .query(), .pipe()\nresult = (\n    df\n    .assign(\n        Bonus=lambda x: x['Salary'] * 0.1,\n        Level=lambda x: pd.cut(x['Experience'], bins=[0, 2, 4, 10], labels=['Junior', 'Mid', 'Senior'])\n    )\n    .query('Salary > 50000')\n    .sort_values('Salary', ascending=False)\n    .groupby('Level')\n    .agg({'Salary': 'mean', 'Bonus': 'sum', 'Name': 'count'})\n    .rename(columns={'Name': 'Count'})\n    .reset_index()\n)\nprint('Chained operation result:')\nprint(result)\nprint()\n\n# Pipe for reusable operations\ndef add_bonus(df, rate=0.1):\n    return df.assign(Bonus=df['Salary'] * rate)\n\ndef categorize_experience(df):\n    return df.assign(Level=pd.cut(df['Experience'], bins=[0, 2, 4, 10], labels=['Junior', 'Mid', 'Senior']))\n\ndef filter_senior(df):\n    return df[df['Level'] == 'Senior']\n\nresult_pipe = (df\n    .pipe(add_bonus, rate=0.15)\n    .pipe(categorize_experience)\n    .pipe(filter_senior)\n)\nprint('Pipe result:')\nprint(result_pipe)

In [ ]:
# Debugging chains\n# method 1: Use .pipe() to print intermediate state\ndef debug_step(df, msg=''):\n    print(f'[DEBUG] {msg}: shape={df.shape}')\n    return df\n\ndebug_result = (\n    df\n    .pipe(debug_step, msg='Original')\n    .assign(Bonus=lambda x: x['Salary'] * 0.1)\n    .pipe(debug_step, msg='After Bonus')\n    .query('Salary > 50000')\n    .pipe(debug_step, msg='After filter')\n)\nprint()\n\n# method 2: Use .loc in the middle of chain\nresult_mid = (df\n    .assign(Bonus=lambda x: x['Salary'] * 0.1)\n    .loc[lambda x: x['Salary'] > 50000]\n    .sort_values('Bonus', ascending=False)\n)\nprint('Mid-chain filtering with .loc:')\nprint(result_mid)

---\n## 11. Working with Large Datasets (Dask, Modin, Vaex)\n\nwhen dataset RAM in fit not be to this libraries use do are. this Pandas API to parallelize do are.\n

In [ ]:
# Dask DataFrame basics (parallel Pandas)\ntry:\n    import dask.dataframe as dd\n    print('Dask available')\n    \n    # Create Dask DataFrame from CSV\n    ddf = dd.read_csv('large_file.csv', blocksize='5MB')\n    print(f'Dask DataFrame: {ddf.npartitions} partitions')\n    print(ddf.compute().head())\n    print()\n    \n    # Lazy operations (not computed until .compute())\n    result = ddf.groupby(ddf['A'] % 10).B.mean()\n    print('Computed result:')\n    print(result.compute())\nexcept ImportError:\n    print('Dask not installed. install: pip install dask')

In [ ]:
# Modin basics (accelerated Pandas)\ntry:\n    import modin.pandas as mpd\n    print('Modin available')\n    \n    # Modin uses Ray/Dask for parallel execution\n    # mdf = mpd.read_csv('large_file.csv')\n    # print(mdf.describe())\nexcept ImportError:\n    print('Modin not installed. install: pip install modin[all]')

In [ ]:
# Vaex basics (out-of-core DataFrames)\ntry:\n    import vaex\n    print('Vaex available')\n    \n    # Convert to Vaex\n    # vdf = vaex.from_pandas(df)\n    # print(vdf.head())\n    #\n    # # Vaex operations are lazy too\n    # result = vdf.groupby(vdf.Category).agg({'value': 'mean'})\nexcept ImportError:\n    print('Vaex not installed. install: pip install vaex')

---\n## 12. Advanced Practice Projects\n

### Project 1: End-to-End Data Analysis Pipeline\n\n**Problem:** Ek comprehensive pipeline create that data load kare, clean kare, feature engineering kare, analysis kare, and report generate kare.\n

In [ ]:
# Project 1: Complete ETL + Analysis Pipeline\nnp.random.seed(42)\nprint('='*60)\nprint('STARTING ETL PIPELINE')\nprint('='*60)\nprint()\n\n# Step 1: Generate raw data\nprint('Step 1: Generating synthetic data...')\nn = 10000\nraw_data = pd.DataFrame({\n    'Transaction_ID': range(1, n+1),\n    'Date': pd.date_range('2023-01-01', periods=n, freq='h')[:n],\n    'Product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Watch', 'Headphones'], n,\n                                 p=[0.2, 0.3, 0.2, 0.15, 0.15]),\n    'Category': np.random.choice(['Electronics', 'Accessories', 'Premium'], n,\n                                 p=[0.5, 0.3, 0.2]),\n    'Quantity': np.random.poisson(2, n).clip(1, 10),\n    'Unit_Price': np.random.choice([25000, 15000, 8000, 5000, 3000], n),\n    'Customer_ID': np.random.randint(1000, 1100, n),\n    'Region': np.random.choice(['North', 'South', 'East', 'West'], n),\n    'Payment_Method': np.random.choice(['Card', 'Cash', 'Online'], n, p=[0.4, 0.3, 0.3])\n})\nprint(f'Generated {n} records')\nprint()\n\n# Step 2: Introduce realistic issues\nprint('Step 2: Introducing data quality issues...')\nraw_data.loc[np.random.choice(n, 200), 'Unit_Price'] = np.nan\nraw_data.loc[np.random.choice(n, 150), 'Customer_ID'] = -1  # Invalid\nraw_data.loc[np.random.choice(n, 50), 'Region'] = 'Invalid'  # Bad data\nprint(f'Missing prices: {raw_data["Unit_Price"].isna().sum()}')\nprint(f'Invalid customers: {(raw_data["Customer_ID"] == -1).sum()}')\nprint(f'Invalid regions: {(raw_data["Region"] == "Invalid").sum()}')\nprint()\n\n# Step 3: Data cleaning\nprint('Step 3: Cleaning data...')\nclean = raw_data.copy()\n\n# Fix missing prices with median per product\nclean['Unit_Price'] = clean.groupby('Product')['Unit_Price'].transform(\n    lambda x: x.fillna(x.median()))\n\n# Fix invalid customers\nclean['Customer_ID'] = clean['Customer_ID'].replace(-1, np.nan)\nclean['Customer_ID'] = clean.groupby('Region')['Customer_ID'].transform(\n    lambda x: x.fillna(x.median()))\n\n# Fix invalid regions (mode imputation)\nvalid_regions = ['North', 'South', 'East', 'West']\nclean.loc[~clean['Region'].isin(valid_regions), 'Region'] = clean['Region'].mode()[0]\n\n# Remove duplicates\nclean = clean.drop_duplicates()\nprint(f'Cleaned data shape: {clean.shape}')\nprint()\n\n# Step 4: Feature engineering\nprint('Step 4: Feature engineering...')\nclean['Revenue'] = clean['Quantity'] * clean['Unit_Price']\nclean['Hour'] = clean['Date'].dt.hour\nclean['DayOfWeek'] = clean['Date'].dt.day_name()\nclean['Month'] = clean['Date'].dt.month\nclean['Is_Weekend'] = clean['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)\nclean['Revenue_per_Unit'] = clean['Revenue'] / clean['Quantity']\nclean['Log_Revenue'] = np.log1p(clean['Revenue'])\nprint('Features added')

In [ ]:
# Step 5: Analysis\nprint('Step 5: Analysis...')\nprint()\n\n# Product performance\nproduct_analysis = (\n    clean.groupby('Product')\n    .agg(\n        Total_Revenue=('Revenue', 'sum'),\n        Avg_Revenue=('Revenue', 'mean'),\n        Total_Qty=('Quantity', 'sum'),\n        Transactions=('Transaction_ID', 'count'),\n        Avg_Price=('Unit_Price', 'mean')\n    )\n    .assign(Revenue_Pct=lambda x: x['Total_Revenue'] / x['Total_Revenue'].sum() * 100)\n    .sort_values('Total_Revenue', ascending=False)\n)\nprint('=== Product Performance ===')\nprint(product_analysis)\nprint()\n\n# Regional analysis\nregional = clean.groupby('Region')['Revenue'].agg(['sum', 'mean', 'count'])\nprint('=== Regional Analysis ===')\nprint(regional)\nprint()\n\n# Hourly patterns\nhourly = clean.groupby('Hour')['Revenue'].mean()\nprint('=== Peak Hours (Top 3) ===')\nprint(hourly.nlargest(3))\nprint()\n\n# Payment method analysis\npayment = clean.groupby('Payment_Method').agg(\n    Revenue=('Revenue', 'sum'),\n    Transactions=('Transaction_ID', 'count')\n).assign(Avg_Transaction=lambda x: x['Revenue'] / x['Transactions'])\nprint('=== Payment Analysis ===')\nprint(payment)

In [ ]:
# Step 6: Visualization\nfig, axes = plt.subplots(2, 3, figsize=(15, 10))\n\n# Revenue by product\nproduct_analysis['Total_Revenue'].plot(kind='bar', ax=axes[0, 0], color='skyblue')\naxes[0, 0].set_title('Revenue by Product')\naxes[0, 0].tick_params(axis='x', rotation=45)\n\n# Hourly revenue trend\nhourly.plot(ax=axes[0, 1], marker='o', color='green')\naxes[0, 1].set_title('Hourly Revenue Trend')\n\n# Regional pie\nregional['sum'].plot(kind='pie', autopct='%1.1f%%', ax=axes[0, 2])\naxes[0, 2].set_title('Revenue by Region')\naxes[0, 2].set_ylabel('')\n\n# Revenue distribution\nclean['Revenue'].hist(bins=50, ax=axes[1, 0], edgecolor='black')\naxes[1, 0].set_title('Revenue Distribution')\n\n# Day of week analysis\ndow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']\ndow = clean.groupby('DayOfWeek')['Revenue'].mean().reindex(dow_order)\ndow.plot(kind='line', marker='s', ax=axes[1, 1], color='red')\naxes[1, 1].set_title('Avg Revenue by Day')\naxes[1, 1].tick_params(axis='x', rotation=45)\n\n# Quantity vs Revenue scatter\nclean.sample(500).plot(x='Quantity', y='Revenue', kind='scatter', alpha=0.5, ax=axes[1, 2])\naxes[1, 2].set_title('Quantity vs Revenue')\n\nplt.tight_layout()\nplt.show()

In [ ]:
# Step 7: Generate summary report\nprint('='*60)\nprint('FINAL BUSINESS SUMMARY REPORT')\nprint('='*60)\nprint()\n\ntotal_revenue = clean['Revenue'].sum()\ntotal_transactions = len(clean)\navg_order_value = total_revenue / total_transactions\ntop_product = product_analysis.index[0]\ntop_region = regional['sum'].idxmax()\npeak_hour = hourly.idxmax()\n\nprint(f'Total Revenue: PKR {total_revenue:,.0f}')\nprint(f'Total Transactions: {total_transactions:,}')\nprint(f'Average Order value: PKR {avg_order_value:,.0f}')\nprint(f'Top Product: {top_product}')\nprint(f'Top Region: {top_region}')\nprint(f'Peak Business Hour: {peak_hour}:00')\nprint(f'Weekend Sales: {clean["Is_Weekend"].mean()*100:.1f}% of total')\nprint(f'Most Common Payment: {payment.index[0]}')\nprint(f'Average Quantity per Transaction: {clean["Quantity"].mean():.1f}')\nprint()\nprint('='*60)\nprint('PIPELINE COMPLETED SUCCESSFULLY')

### Project 2: Large Dataset Optimization\n\n**Problem:** Ek million-row dataset generate do, memory optimize do, and operations benchmark do.\n

In [ ]:
# Project 2: Large Data Optimization\nprint('Generating 1M row dataset...')\nn = 1_000_000\n\nstart = time.time()\ndf_large = pd.DataFrame({\n    'ID': range(n),\n    'Category': np.random.choice(['A', 'B', 'C', 'D', 'E'], n),\n    'value': np.random.randn(n),\n    'Group': np.random.choice(['X', 'Y', 'Z'], n),\n    'Date': pd.date_range('2020-01-01', periods=n, freq='s')[:n],\n    'Flag': np.random.choice([True, False], n),\n    'Score': np.random.randint(0, 100, n),\n    'City': np.random.choice(['Karachi', 'Lahore', 'Islamabad', 'Peshawar', 'Quetta', 'Multan', 'Faisalabad', 'Rawalpindi'], n),\n    'Category2': np.random.choice(['Type1', 'Type2', 'Type3', 'Type4', 'Type5', 'Type6', 'Type7', 'Type8'], n)\n})\nprint(f'Data generation: {time.time() - start:.2f} sec')\nprint(f'Memory: {df_large.memory_usage(deep=True).sum()/1024**3:.2f} GB')\nprint()\n\n# Optimize\nstart = time.time()\ndf_opt = df_large.copy()\n\n# Downcast numeric\ndf_opt['ID'] = pd.to_numeric(df_opt['ID'], downcast='integer')\ndf_opt['Score'] = pd.to_numeric(df_opt['Score'], downcast='integer')\ndf_opt['value'] = pd.to_numeric(df_opt['value'], downcast='float')\n\n# Convert to categorical\ndf_opt['Category'] = df_opt['Category'].astype('category')\ndf_opt['Group'] = df_opt['Group'].astype('category')\ndf_opt['Flag'] = df_opt['Flag'].astype('bool')\ndf_opt['City'] = df_opt['City'].astype('category')\ndf_opt['Category2'] = df_opt['Category2'].astype('category')\n\nopt_time = time.time() - start\nprint(f'Optimization: {opt_time:.2f} sec')\nprint(f'Optimized memory: {df_opt.memory_usage(deep=True).sum()/1024**3:.3f} GB')\nprint(f'Saved: {100*(1 - df_opt.memory_usage(deep=True).sum()/df_large.memory_usage(deep=True).sum()):.0f}%')\nprint()\n\n# Benchmark operations\nprint('Benchmarking operations...')\n\nstart = time.time()\ndf_opt.groupby('Category')['value'].mean()\nprint(f'GroupBy mean: {time.time() - start:.3f} sec')\n\nstart = time.time()\ndf_opt.groupby(['Category', 'Group']).agg({'value': ['mean', 'std'], 'Score': 'sum'})\nprint(f'Multi-groupBy: {time.time() - start:.3f} sec')\n\nstart = time.time()\ndf_opt.query('Category == "A" and value > 0')\nprint(f'Query filter: {time.time() - start:.3f} sec')\n\nstart = time.time()\ndf_opt['value'].rolling(100).mean()\nprint(f'Rolling 100 mean: {time.time() - start:.3f} sec')

### Project 3: Complex Data Transformation\n\n**Problem:** Nested JSON jesa data lo, use flatten do, multiple transformations do, and aggregated report create.\n

In [ ]:
# Project 3: Complex Transformation\n# Simulate nested/JSON data\ndata = []\nfor i in range(1000):\n    data.append({\n        'order_id': f'ORD{i:05d}',\n        'customer': {\n            'name': np.random.choice(['Ali', 'Sara', 'Ahmed', 'Fatima']),\n            'tier': np.random.choice(['Gold', 'Silver', 'Bronze'], p=[0.2, 0.3, 0.5]),\n            'address': {\n                'city': np.random.choice(['Karachi', 'Lahore', 'Islamabad']),\n                'area': np.random.choice(['Area1', 'Area2', 'Area3'])\n            }\n        },\n        'items': [\n            {'product': p, 'qty': np.random.randint(1, 5), 'price': np.random.randint(100, 1000)}\n            for p in np.random.choice(['A', 'B', 'C'], np.random.randint(1, 4))\n        ],\n        'payment': {\n            'method': np.random.choice(['Card', 'Cash']),\n            'status': np.random.choice(['Completed', 'Pending', 'Failed'], p=[0.8, 0.15, 0.05])\n        }\n    })\n\nprint(f'Total orders: {len(data)}')\nprint()\n\n# Step 1: Flatten the nested structure\ndef flatten_order(order):\n    rows = []\n    for item in order['items']:\n        rows.append({\n            'order_id': order['order_id'],\n            'customer_name': order['customer']['name'],\n            'customer_tier': order['customer']['tier'],\n            'city': order['customer']['address']['city'],\n            'area': order['customer']['address']['area'],\n            'product': item['product'],\n            'qty': item['qty'],\n            'price': item['price'],\n            'total_item': item['qty'] * item['price'],\n            'payment_method': order['payment']['method'],\n            'payment_status': order['payment']['status']\n        })\n    return rows\n\nstart = time.time()\nflat_data = []\nfor order in data:\n    flat_data.extend(flatten_order(order))\ndf_flat = pd.DataFrame(flat_data)\nprint(f'Step 1: Flattened to {len(df_flat)} rows in {time.time()-start:.2f} sec')\nprint(df_flat.head())\nprint()\n\n# Step 2: Transformations\ndf_trans = (\n    df_flat\n    .assign(\n        order_total=lambda x: x.groupby('order_id')['total_item'].transform('sum'),\n        is_big_order=lambda x: x['order_total'] > 5000,\n        discount=lambda x: np.where(x['customer_tier'] == 'Gold', 0.1,\n                                    np.where(x['customer_tier'] == 'Silver', 0.05, 0)),\n        final_price=lambda x: x['total_item'] * (1 - x['discount'])\n    )\n)\nprint('Step 2: Transformations applied')\nprint()\n\n# Step 3: Analysis\nprint('Step 3: Analysis...')\nprint()\n\n# Tier analysis\ntier_analysis = df_trans.groupby('customer_tier').agg({\n    'order_id': 'nunique',\n    'final_price': ['sum', 'mean'],\n    'qty': 'sum'\n})\nprint('Customer tier analysis:')\nprint(tier_analysis)\nprint()\n\n# Product popularity\nproduct_pop = df_trans.groupby('product')['qty'].sum().sort_values(ascending=False)\nprint('Product popularity:')\nprint(product_pop)\nprint()\n\n# Payment status analysis\npayment_status = df_trans.groupby('payment_status').size()\nprint('Payment status distribution:')\nprint(payment_status)

In [ ]:
# Step 4: Cross-tabulation and pivot\n# City x Product cross-sell\ncross = pd.crosstab(df_trans['city'], df_trans['product'],\n                     values=df_trans['qty'], aggfunc='sum',\n                     margins=True, margins_name='Total')\nprint('City x Product sales matrix:')\nprint(cross)\nprint()\n\n# Pivot: Average order value by city and tier\npivot_avg = pd.pivot_table(df_trans, values='final_price',\n                            index='city', columns='customer_tier',\n                            aggfunc='mean', margins=True)\nprint('Average order value (City x Tier):')\nprint(pivot_avg)\nprint()\n\n# Step 5: Export results\nprint('Step 5: Exporting results...')\nresults = {\n    'tier_analysis.csv': tier_analysis.reset_index(),\n    'product_popularity.csv': product_pop.reset_index(),\n    'cross_sell.csv': cross.reset_index(),\n    'city_tier_pivot.csv': pivot_avg.reset_index()\n}\nfor name, df_out in results.items():\n    df_out.to_csv(name, index=False)\n    print(f'  Saved {name}')\nprint()\nprint('Complex transformation project completed!')

---\n## Congratulations! 🎉\n\nAapne Pandas Advance successfully complete do taken! Ab you:- Performance optimization techniques know are (vectorization, numba, raw=True)\n- Memory optimization (downcasting, categorical, sparse, chunking) use can\n- Advanced reshaping (wide_to_long, explode, cross-join) know are\n- Advanced I/O (Parquet, Feather, HDF5) formats use can\n- Complex time series analysis can\n- Styler and conditional formatting use can\n- MultiIndex advanced operations know are\n- Large datasets Dask/Modin with handle can\n- End-to-end data pipelines make can are\n\n**this Pandas of complete journey tha!** Ab you kisi also real-world data analysis project for ready are.\n